In [2]:
import cv2
import pickle
import os
from deepface import DeepFace
import numpy as np
import time
from PIL import Image

24-10-30 17:51:31 - Directory /root/.deepface has been created
24-10-30 17:51:31 - Directory /root/.deepface/weights has been created


In [1]:
!pip install deepface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.8 MB/s eta 0:00:00:00:01
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114248 sha256=24b31508336a0d2270609d33248946a8e102165a7dd546773e9ad09dad5a3f8d
  Stored in directory: /root/.cache/pip/wheels/19/39/2f/2d3cadc408a8804103f1c34ddd4b9f6a93497b11fa96fe738e
Successfully built fire


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline


cap = cv2.VideoCapture('/kaggle/input/53-bm-final/final_test.mp4')

while True:
    ret,img = cap.read()

    imgplot = plt.imshow(img)
    plt.show()
    plt.close()
    

In [4]:
# Dummy GPU test
import tensorflow as tf
with tf.device('/GPU:0'):
    for _ in range(10):
        tf.random.normal([1000, 1000, 1000]).numpy()


In [6]:

import cv2
import pickle
import os
from deepface import DeepFace
import numpy as np




def normalize_resize_image(img, size=(160,160)):
    """
    Resizes and normalizes the image.
    """
    img = cv2.resize(img, size)
    img = img / 255.0  # Normalize pixel values to [0, 1]
    return img


def encode_faces(dataset_path, encoding_path):
    """
    Encodes faces in the custom dataset and saves them as a pickle file.
    Each subdirectory in `dataset_path` should be named after the person it contains images of.
    """
    if os.path.exists(encoding_path):
        # Load encodings from pickle file
        with open(encoding_path, 'rb') as file:
            return pickle.load(file)
    else:
        encodings = {}
        for person_name in os.listdir(dataset_path):
            
            person_dir = os.path.join(dataset_path, person_name)

            if not os.path.isdir(person_dir):
                continue
            
            
            person_encodings = []
            for image_name in os.listdir(person_dir):
              
                img_path = os.path.join(person_dir, image_name)
                img =cv2.imread(img_path)
                # img = normalize_resize_image(img)
                

                print(img_path)
                try:
                    # Encode the face using DeepFace's ArcFace model
                    embedding = DeepFace.represent(img_path=img, model_name='Facenet',normalization='Facenet', enforce_detection=False)
                    person_encodings.append(embedding)
                except Exception as e:
                    print(f"Could not process image {img_path}: {e}")
            
            # Store all encodings for the person
            if person_encodings:
                encodings[person_name] = person_encodings
        
        # Save encodings to file
        with open(encoding_path, 'wb') as f:
            pickle.dump(encodings, f)
            f.close()
        print("Encodings saved to", encoding_path)
        
        with open(encoding_path, 'rb') as file:
            return pickle.load(file)
            file.close()
        


In [7]:

# Paths
DATASET_PATH = "/kaggle/input/53-bm-final/53BM-face/53BM-face"  # Update with your dataset path
ENCODINGS_PATH = "53bm.pkl"  # File to save encodings


def cosine_similarity(embedding1, embedding2):
    """
    Calculates cosine similarity between two embeddings.
    """
    dot_product = np.dot(embedding1, embedding2)
    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)
    return dot_product / (norm1 * norm2)

def euclidean_distance(embedding1, embedding2):
    """
    Calculates Euclidean distance between two embeddings.
    """
    return np.linalg.norm(np.array(embedding1) - np.array(embedding2))

def euclidean_l2_distance(embedding1, embedding2):
    """
    Calculates Euclidean distance after L2-normalizing both embeddings.
    """
    embedding1_normalized = embedding1 / np.linalg.norm(embedding1)
    embedding2_normalized = embedding2 / np.linalg.norm(embedding2)
    return np.linalg.norm(embedding1_normalized - embedding2_normalized)

def recognize_face(frame, encodings, threshold=0.01):
    """
    Recognizes faces in a frame by comparing with known encodings using Euclidean distance.
    """

    try:
        # Resize and normalize the frame
        # frame = normalize_resize_image(frame)
        
        # Get the face embedding of the input frame
        frame_embedding = DeepFace.represent(frame,model_name='Facenet', enforce_detection=False,normalization="Facenet",)[0]['embedding']

        global recognized_faces
        recognized_faces = []
        for person_name, known_encodings in encodings.items():

            print()
            for known_embedding in known_encodings:
                # Calculate Euclidean distance]

                distance = cosine_similarity(np.array(known_embedding[0]['embedding']),np.array(frame_embedding))

                print(person_name,distance)
                # if distance > threshold:
                #     recognized_faces.append(person_name)
                #     break
                
                if distance > threshold:
                    
                    recognized_faces.append((distance,person_name))
                    
        recognized_faces = sorted(recognized_faces, key=lambda x: x[0])
        
        print("=================")
    except Exception as e:
        print(f"Could not process frame for recognition: {e}")
    
    return 'o' if len(recognized_faces)==0 else recognized_faces[-1] 
    # return recognized_faces

def process_video(video_path, encodings, output_path="output.mp4"):
    """
    Processes a video to recognize faces and draw bounding boxes with names.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return
    mask = cv2.imread('/kaggle/input/53-bm-final/mask__.png')
    
#     Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, 30, (width, height))
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        f=0
#         frame = cv2.resize(frame,(480,640))
#         mask = cv2.resize(mask,(480,640))
        img_region = cv2.bitwise_and(frame,mask)
        # Detect faces in the frame
        face_locations = DeepFace.extract_faces(img_region, detector_backend='ssd', enforce_detection=False,align=False)
        # print(face_locations)
        if len(face_locations) > 0:
            # Loop over detected faces
            for face_data in face_locations:
                facial_area = face_data['facial_area']
                x = facial_area['x']
                y = facial_area['y']
                w = facial_area['w']
                h = facial_area['h']
                
                # Optionally, check if 'left_eye' and 'right_eye' exist
                leye = face_data.get('left_eye')
                reye = face_data.get('right_eye')
                
                face = img_region[y:y+h, x:x+w]

                
              
                # Recognize the face
                recognized_faces = recognize_face(face, encodings)
                if len(recognized_faces)==1:
#                     cv2.imshow('test',frame) 
#                     cv2.waitKey(0)
                    out.write(frame)
#                     imgplot = plt.imshow(frame)
#                     plt.show()
#                     plt.close()
                    
                    f=1
                else:
                    name = recognized_faces[1]  # Take the first recognized name
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    cv2.putText(frame, name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        # print(recognized_faces[1])
        # Write the processed frame to the output video
        
        if f==0:
            out.write(frame)
#             imgplot = plt.imshow(frame)
#             plt.show()
#             plt.close()
#             cv2.imshow('test',frame)
#             cv2.waitKey(0)
    # Release resources
    cap.release()
    out.release()
#     cv2.destroyAllWindows()
    
    # return recognized_faces
    print(f"Output video saved to {output_path}")


# Load encodings
encode_dict = encode_faces(DATASET_PATH, ENCODINGS_PATH)


# Process the video and save with bounding boxes and names
video_path = "/kaggle/input/53-bm-final/final_test.mp4"  # Update with your input video file
output_path = "trheda.mp4"
process_video(video_path, encode_dict, output_path)



24-10-30 17:58:18 - deploy.prototxt will be downloaded...


Downloading...
From: https://github.com/opencv/opencv/raw/3.4.0/samples/dnn/face_detector/deploy.prototxt
To: /root/.deepface/weights/deploy.prototxt
28.1kB [00:00, 40.8MB/s]                   


24-10-30 17:58:18 - res10_300x300_ssd_iter_140000.caffemodel will be downloaded...


Downloading...
From: https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
To: /root/.deepface/weights/res10_300x300_ssd_iter_140000.caffemodel
100%|██████████| 10.7M/10.7M [00:00<00:00, 177MB/s]


24-10-30 17:58:22 - facenet_weights.h5 will be downloaded...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5
100%|██████████| 92.2M/92.2M [00:00<00:00, 112MB/s] 



anas -0.011873746365560984

mahi 0.22464286749988172

mayeen 0.0642537281081565

jobayer -0.024494817831983085

nahian -0.01833505140785725

sayem -0.036959787881420486

durjoy 0.05982369006903738

kashfi 0.14640656068849114

arman 0.1733751856992234

tasin 0.18962240471295583

nahid 0.025435268180346297

chahodul 0.11187395471354163

raihan 0.11360652294222807

tawheed -0.07196212037098375

miftah 0.14925829701765775

rayhan 0.007602576809884556

saim 0.1509720098259279

ahnab 0.15558900179899202

niloy 0.08013015585168527

amir 0.07277430012122635

rafi 0.28560464803108715

faisal 0.07127406728515209

sajib 0.07968207616493528

mamun 0.18273998467072664

tamim 0.001015106414865583

fardin 0.12028146029552243

rakib 0.1864090936437107

parba 0.11640561314039449

atik 0.08697248012645767

tanvir 0.15289469212897477

anas -0.012665953114540522

mahi 0.22353890473818727

mayeen 0.06569928107356389

jobayer -0.02433395983907144

nahian -0.0183240437781365

sayem -0.036018424757918194

du